# User-to-Item Recommendation System with TF-IDF and NER
In this notebook we created a recsys based on TF-IDF but with the addition of NER (named entity recognition).

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
wines_features = pd.read_csv(
    Path("../data/processed/wines_features.csv"),
    usecols=[
        "WineID", "WineName", "Type", "Country", "RegionName",
        "Body", "Acidity", "Grapes", "Harmonize", "bert_text"
    ]
)

ratings = pd.read_csv(
    Path("../dataset/XWines_Full_21M_ratings.csv"),
    usecols=["UserID", "WineID", "Rating"],
    dtype={"UserID": "int32", "WineID": "int32", "Rating": "float32"}
)

wines_features = wines_features.drop_duplicates(subset="WineID").copy()
wines_features = wines_features[wines_features["bert_text"].notna()].copy()
wines_features = wines_features[wines_features["bert_text"].str.strip() != ""].copy()

wines_features["WineID"] = wines_features["WineID"].astype("int32")
ratings["WineID"] = ratings["WineID"].astype("int32")
ratings = ratings.drop_duplicates(subset=["UserID", "WineID"]).copy()

print("wines_features:", wines_features.shape)
print("ratings:", ratings.shape)

wines_features: (100646, 10)
ratings: (20590800, 3)


In [3]:
import ast

def safe_parse_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("[") and x.endswith("]"):
            try:
                return ast.literal_eval(x)
            except Exception:
                return []
    return []

wines_features["Grapes"] = wines_features["Grapes"].apply(safe_parse_list)
wines_features["Harmonize"] = wines_features["Harmonize"].apply(safe_parse_list)

In [4]:
POSITIVE_THRESHOLD = 4.0
MIN_POSITIVE_INTERACTIONS = 5
MAX_POSITIVE_INTERACTIONS = 200
MAX_USERS = 5000
RANDOM_STATE = 42
TEST_FRACTION = 0.2
SAMPLED_USERS_PATH = Path("../data/results/user_to_item_sampled_users_5000.csv")

positive = ratings[ratings["Rating"] >= POSITIVE_THRESHOLD].copy()
user_counts = positive["UserID"].value_counts()
eligible_users = user_counts[
    (user_counts >= MIN_POSITIVE_INTERACTIONS) &
    (user_counts <= MAX_POSITIVE_INTERACTIONS)
].index
positive = positive[positive["UserID"].isin(eligible_users)].copy()

if SAMPLED_USERS_PATH.exists():
    sampled_users = pd.read_csv(SAMPLED_USERS_PATH)["UserID"].to_numpy()
else:
    rng = np.random.default_rng(RANDOM_STATE)
    sampled_users = rng.choice(
        positive["UserID"].unique(),
        size=min(MAX_USERS, positive["UserID"].nunique()),
        replace=False
    )
    SAMPLED_USERS_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.Series(sampled_users).to_csv(SAMPLED_USERS_PATH, index=False, header=["UserID"])

positive = positive[positive["UserID"].isin(sampled_users)].copy()

print("users used in this run:", positive["UserID"].nunique())
print("positive interactions used:", positive.shape)

users used in this run: 5000
positive interactions used: (70478, 3)


Similarly to other notebooks we used the same positive feedback threshold for a fair comparison between the recsys models.

In [5]:
train_parts = []
test_parts = []

for user_id, group in positive.groupby("UserID"):
    group = group.sample(frac=1, random_state=RANDOM_STATE)
    n_test = max(1, int(np.ceil(len(group) * TEST_FRACTION)))

    test_group = group.iloc[:n_test]
    train_group = group.iloc[n_test:]

    if len(train_group) == 0:
        continue

    train_parts.append(train_group)
    test_parts.append(test_group)

train_pos = pd.concat(train_parts, ignore_index=True)
test_pos = pd.concat(test_parts, ignore_index=True)

print("train_pos:", train_pos.shape)
print("test_pos:", test_pos.shape)

train_pos: (54410, 3)
test_pos: (16068, 3)


In [6]:
def normalize_entity(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r"[/\-]", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace(" ", "_")
    return text

grape_vocab = sorted({
    normalize_entity(g)
    for row in wines_features["Grapes"]
    for g in row
    if isinstance(g, str) and g.strip()
})

food_vocab = sorted({
    normalize_entity(h)
    for row in wines_features["Harmonize"]
    for h in row
    if isinstance(h, str) and h.strip()
})

country_vocab = sorted(wines_features["Country"].dropna().astype(str).map(normalize_entity).unique())
region_vocab = sorted(wines_features["RegionName"].dropna().astype(str).map(normalize_entity).unique())
type_vocab = sorted(wines_features["Type"].dropna().astype(str).map(normalize_entity).unique())
body_vocab = sorted(wines_features["Body"].dropna().astype(str).map(normalize_entity).unique())
acidity_vocab = sorted(wines_features["Acidity"].dropna().astype(str).map(normalize_entity).unique())

print("grapes:", len(grape_vocab))
print("foods:", len(food_vocab))
print("countries:", len(country_vocab))
print("regions:", len(region_vocab))

grapes: 777
foods: 66
countries: 62
regions: 2160


In [7]:
def find_vocab_matches(text, vocab):
    text = normalize_entity(text)
    matches = [term for term in vocab if term and term in text]
    return sorted(set(matches))

def extract_entities(row):
    text = str(row["bert_text"])

    grapes = sorted(set([normalize_entity(g) for g in row["Grapes"] if isinstance(g, str) and g.strip()]))
    foods = sorted(set([normalize_entity(h) for h in row["Harmonize"] if isinstance(h, str) and h.strip()]))

    country = normalize_entity(str(row["Country"])) if pd.notna(row["Country"]) else ""
    region = normalize_entity(str(row["RegionName"])) if pd.notna(row["RegionName"]) else ""
    wine_type = normalize_entity(str(row["Type"])) if pd.notna(row["Type"]) else ""
    body = normalize_entity(str(row["Body"])) if pd.notna(row["Body"]) else ""
    acidity = normalize_entity(str(row["Acidity"])) if pd.notna(row["Acidity"]) else ""

    # optional text matching fallback
    grapes_from_text = find_vocab_matches(text, grape_vocab)
    foods_from_text = find_vocab_matches(text, food_vocab)

    all_grapes = sorted(set(grapes + grapes_from_text))
    all_foods = sorted(set(foods + foods_from_text))

    return {
        "grapes": all_grapes,
        "foods": all_foods,
        "country": country,
        "region": region,
        "type": wine_type,
        "body": body,
        "acidity": acidity
    }

In [8]:
entity_data = wines_features.apply(extract_entities, axis=1)

wines_features["ner_grapes"] = entity_data.apply(lambda x: x["grapes"])
wines_features["ner_foods"] = entity_data.apply(lambda x: x["foods"])
wines_features["ner_country"] = entity_data.apply(lambda x: x["country"])
wines_features["ner_region"] = entity_data.apply(lambda x: x["region"])
wines_features["ner_type"] = entity_data.apply(lambda x: x["type"])
wines_features["ner_body"] = entity_data.apply(lambda x: x["body"])
wines_features["ner_acidity"] = entity_data.apply(lambda x: x["acidity"])

In [9]:
def build_ner_text(row):
    tokens = []

    if row["ner_type"]:
        tokens.append(f"type_{row['ner_type']}")
    if row["ner_country"]:
        tokens.append(f"country_{row['ner_country']}")
    if row["ner_region"]:
        tokens.append(f"region_{row['ner_region']}")
    if row["ner_body"]:
        tokens.append(f"body_{row['ner_body']}")
    if row["ner_acidity"]:
        tokens.append(f"acidity_{row['ner_acidity']}")

    tokens.extend([f"grape_{g}" for g in row["ner_grapes"] if g])
    tokens.extend([f"food_{f}" for f in row["ner_foods"] if f])

    return " ".join(tokens)

wines_features["ner_text"] = wines_features.apply(build_ner_text, axis=1)

We created a structured representation focused on certain wine attributes including the wine type, body, acidity, grapes and wine pairing with the goal of reducing noise from the original text and emphasizing the features that are more directly relevant for recommendations.

In [10]:
wines_features[["WineName", "bert_text", "ner_text"]].head(5)

,WineName,bert_text,ner_text
0,Espumante Moscatel,espumante moscatel. This wine is a medium bodi...,type_sparkling country_brazil region_serra_gaú...
1,Ancellotta,"ancellotta. This wine is a medium bodied, medi...",type_red country_brazil region_serra_gaúcha bo...
2,Cabernet Sauvignon,cabernet sauvignon. This wine is a full bodied...,type_red country_brazil region_serra_gaúcha bo...
3,Virtus Moscato,"virtus moscato. This wine is a medium bodied, ...",type_white country_brazil region_serra_gaúcha ...
4,Maison de Ville Cabernet-Merlot,maison de ville cabernet merlot. This wine is ...,type_red country_brazil region_serra_gaúcha bo...


In [11]:
tfidf_ner = TfidfVectorizer(min_df=5, max_df=0.8, max_features=30000)
tfidf_ner_matrix = tfidf_ner.fit_transform(wines_features["ner_text"])

In [12]:
wines_features = wines_features.reset_index(drop=True)

wineid_to_idx = pd.Series(wines_features.index, index=wines_features["WineID"]).drop_duplicates()
idx_to_wineid = pd.Series(wines_features["WineID"].values, index=wines_features.index)

train_pos = train_pos[train_pos["WineID"].isin(wineid_to_idx.index)].copy()
test_pos = test_pos[test_pos["WineID"].isin(wineid_to_idx.index)].copy()

print("train_pos after filtering:", train_pos.shape)
print("test_pos after filtering:", test_pos.shape)

train_pos after filtering: (54410, 3)
test_pos after filtering: (16068, 3)


We transformed the structured wine text into TF-IDF vectors so that each wine can be represented numerically.

In [13]:
def build_user_profile_ner(user_history):
    user_history = user_history[user_history["WineID"].isin(wineid_to_idx.index)].copy()

    item_indices = [wineid_to_idx[wine_id] for wine_id in user_history["WineID"]]
    weights = (user_history["Rating"] - 3.0).clip(lower=1.0).to_numpy()

    item_matrix = tfidf_ner_matrix[item_indices]
    weighted_sum = item_matrix.multiply(weights.reshape(-1, 1)).sum(axis=0)
    profile = weighted_sum / weights.sum()

    return csr_matrix(profile)

In [14]:
user_profiles_ner = {}
user_groups = list(train_pos.groupby("UserID"))

for i, (user_id, user_history) in enumerate(user_groups, start=1):
    user_profiles_ner[user_id] = build_user_profile_ner(user_history)
    if i % 500 == 0:
        print(f"Built {i}/{len(user_groups)} NER user profiles")

print("built profiles:", len(user_profiles_ner))

Built 500/5000 NER user profiles
Built 1000/5000 NER user profiles
Built 1500/5000 NER user profiles
Built 2000/5000 NER user profiles
Built 2500/5000 NER user profiles
Built 3000/5000 NER user profiles
Built 3500/5000 NER user profiles
Built 4000/5000 NER user profiles
Built 4500/5000 NER user profiles
Built 5000/5000 NER user profiles
built profiles: 5000


Like in the other notebooks we made user profiles using a weighted combination of vectors of wines the user rated positively.

In [15]:
train_seen = train_pos.groupby("UserID")["WineID"].apply(set).to_dict()
test_relevant = test_pos.groupby("UserID")["WineID"].apply(set).to_dict()

eval_users = sorted(set(user_profiles_ner.keys()) & set(test_relevant.keys()))
print("evaluation users:", len(eval_users))

train_seen_idx = {
    user_id: [wineid_to_idx[w] for w in seen if w in wineid_to_idx]
    for user_id, seen in train_seen.items()
}

evaluation users: 5000


In [16]:
def recommend_user_ner_ids(user_id, top_k=20, candidate_pool=400):
    if user_id not in user_profiles_ner:
        return []

    profile = user_profiles_ner[user_id]
    scores = linear_kernel(profile, tfidf_ner_matrix).ravel()

    seen_idx = train_seen_idx.get(user_id, [])
    if seen_idx:
        scores[seen_idx] = -np.inf

    pool = min(candidate_pool, scores.size)
    candidate_idx = np.argpartition(scores, -pool)[-pool:]
    candidate_idx = candidate_idx[np.argsort(scores[candidate_idx])[::-1]]

    return [int(idx_to_wineid[i]) for i in candidate_idx[:top_k]]

In [17]:
def show_user_recommendations_ner(user_id, top_k=10):
    rec_ids = recommend_user_ner_ids(user_id, top_k=top_k)

    recs = wines_features[wines_features["WineID"].isin(rec_ids)][
        ["WineID", "WineName", "Type", "Country", "RegionName", "Body", "Acidity"]
    ].copy()

    recs["rank"] = recs["WineID"].map({wine_id: i + 1 for i, wine_id in enumerate(rec_ids)})
    recs = recs.sort_values("rank").reset_index(drop=True)

    return recs

The recommendations are generated by comparing each user profile to the wine vectors and ranking wines by cosine similarity.

In [18]:
sample_user = eval_users[0]
show_user_recommendations_ner(sample_user, top_k=10)

,WineID,WineName,Type,Country,RegionName,Body,Acidity,rank
0,190329,Cabernet Sauvignon,Red,United States,Fresno County,Full-bodied,High,1
1,182622,Cabernet Sauvignon,Red,United States,Yolo County,Full-bodied,High,2
2,187110,Single Vineyard Cabernet Sauvignon,Red,United States,Malibu-Newton Canyon,Full-bodied,High,3
3,189128,Los Robles Vineyard Cabernet Sauvignon,Red,United States,Clements Hills,Full-bodied,High,4
4,180192,Cabernet Sauvignon,Red,United States,Moon Mountain District,Full-bodied,High,5
5,165937,Intrépido Reserva Cabernet Sauvignon,Red,United States,Central Valley (USA),Full-bodied,High,6
6,185234,Lava Block Cabernet Sauvignon,Red,United States,Moon Mountain District,Full-bodied,High,7
7,144144,Piave Cabernet,Red,Italy,Piave,Full-bodied,High,8
8,140651,Vino dell'Amicizia Cabernet,Red,Italy,Piave,Full-bodied,High,9
9,151084,Cabernet Sauvignon Lison Pramaggiore,Red,Italy,Lison-Pramaggiore,Full-bodied,High,10


The results for the sample user are highly consistent but we see more flexibility than in the traditional TF-IDF. There is a bigger variety in the region and country of the wine origin.

In [19]:
def precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / k

def recall_at_k(relevant, recommended, k):
    if len(relevant) == 0:
        return 0.0
    recommended_k = recommended[:k]
    hits = sum(1 for item in recommended_k if item in relevant)
    return hits / len(relevant)

def dcg_at_k(relevant, recommended, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            score += 1.0 / np.log2(i + 1)
    return score

def ndcg_at_k(relevant, recommended, k):
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg_at_k(relevant, recommended, k) / ideal_dcg

def average_precision_at_k(relevant, recommended, k):
    recommended_k = recommended[:k]
    hits = 0
    score = 0.0
    for i, item in enumerate(recommended_k, start=1):
        if item in relevant:
            hits += 1
            score += hits / i
    denom = min(len(relevant), k)
    return score / denom if denom > 0 else 0.0

def hit_rate_at_k(relevant, recommended, k):
    return float(any(item in relevant for item in recommended[:k]))

def reciprocal_rank_at_k(relevant, recommended, k):
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            return 1.0 / i
    return 0.0

In [20]:
def evaluate_recommender(recommend_func, users, test_relevant_dict, ks=(5, 10, 20)):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_dict.get(user_id, set())
        if not relevant:
            continue

        recommended = recommend_func(user_id, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [21]:
ner_eval = evaluate_recommender(
    recommend_func=recommend_user_ner_ids,
    users=eval_users,
    test_relevant_dict=test_relevant,
    ks=(5, 10, 20)
)

ner_summary = (
    ner_eval
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="NER TF-IDF")
)

ner_summary

Evaluated 500/5000 users
Evaluated 1000/5000 users
Evaluated 1500/5000 users
Evaluated 2000/5000 users
Evaluated 2500/5000 users
Evaluated 3000/5000 users
Evaluated 3500/5000 users
Evaluated 4000/5000 users
Evaluated 4500/5000 users
Evaluated 5000/5000 users


,NER TF-IDF
Precision@5,0.0008
Recall@5,0.0015
NDCG@5,0.0012
MAP@5,0.0008
HitRate@5,0.0040
MRR@5,0.0017
Precision@10,0.0009
Recall@10,0.0035
NDCG@10,0.0020
MAP@10,0.0010


In [22]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
ner_summary.to_csv("../data/results/ner_summary.csv")

In [23]:
wines_features["eval_signature"] = (
    wines_features["Type"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Country"].astype(str).str.lower().str.strip() + "|" +
    wines_features["RegionName"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Body"].astype(str).str.lower().str.strip() + "|" +
    wines_features["Acidity"].astype(str).str.lower().str.strip()
)

wineid_to_signature = dict(zip(wines_features["WineID"], wines_features["eval_signature"]))

test_relevant_signatures = {
    user_id: {wineid_to_signature[w] for w in wine_ids if w in wineid_to_signature}
    for user_id, wine_ids in test_relevant.items()
}

In [24]:
def recommended_ids_to_signatures(recommended_ids, top_k=None):
    signatures = []
    seen_signatures = set()

    for wine_id in recommended_ids:
        if wine_id not in wineid_to_signature:
            continue

        signature = wineid_to_signature[wine_id]
        if signature in seen_signatures:
            continue

        signatures.append(signature)
        seen_signatures.add(signature)

        if top_k is not None and len(signatures) == top_k:
            break

    return signatures

In [25]:
def evaluate_recommender_soft(recommend_func, users, test_relevant_signatures_dict, ks=(5, 10, 20), oversample_factor=5):
    rows = []
    max_k = max(ks)

    for i, user_id in enumerate(users, start=1):
        relevant = test_relevant_signatures_dict.get(user_id, set())
        if not relevant:
            continue

        recommended_ids = recommend_func(user_id, top_k=max_k * oversample_factor)
        recommended = recommended_ids_to_signatures(recommended_ids, top_k=max_k)

        row = {"UserID": user_id}
        for k in ks:
            row[f"Precision@{k}"] = precision_at_k(relevant, recommended, k)
            row[f"Recall@{k}"] = recall_at_k(relevant, recommended, k)
            row[f"NDCG@{k}"] = ndcg_at_k(relevant, recommended, k)
            row[f"MAP@{k}"] = average_precision_at_k(relevant, recommended, k)
            row[f"HitRate@{k}"] = hit_rate_at_k(relevant, recommended, k)
            row[f"MRR@{k}"] = reciprocal_rank_at_k(relevant, recommended, k)

        rows.append(row)

        if i % 500 == 0:
            print(f"Soft-evaluated {i}/{len(users)} users")

    return pd.DataFrame(rows)

In [26]:
ner_eval_soft = evaluate_recommender_soft(
    recommend_func=recommend_user_ner_ids,
    users=eval_users,
    test_relevant_signatures_dict=test_relevant_signatures,
    ks=(5, 10, 20),
    oversample_factor=5
)

ner_summary_soft = (
    ner_eval_soft
    .drop(columns="UserID")
    .mean()
    .round(4)
    .to_frame(name="NER TF-IDF Soft")
)

ner_summary_soft

Soft-evaluated 500/5000 users
Soft-evaluated 1000/5000 users
Soft-evaluated 1500/5000 users
Soft-evaluated 2000/5000 users
Soft-evaluated 2500/5000 users
Soft-evaluated 3000/5000 users
Soft-evaluated 3500/5000 users
Soft-evaluated 4000/5000 users
Soft-evaluated 4500/5000 users
Soft-evaluated 5000/5000 users


,NER TF-IDF Soft
Precision@5,0.0355
Recall@5,0.0701
NDCG@5,0.0780
MAP@5,0.0578
HitRate@5,0.1694
MRR@5,0.1346
Precision@10,0.0208
Recall@10,0.0797
NDCG@10,0.0802
MAP@10,0.0574


In [27]:
Path("../data/results").mkdir(parents=True, exist_ok=True)
ner_eval_soft.to_csv("../data/results/ner_eval_soft.csv", index=False)
ner_summary_soft.to_csv("../data/results/ner_summary_soft.csv")

# Save the model

In [28]:
from pathlib import Path
import pandas as pd

ARMS_DIR = Path("../bandits/saved_arms")
ARMS_DIR.mkdir(parents=True, exist_ok=True)

def export_arm_recs(recommend_fn, users, out_csv, top_k=100):
    rows = []
    for uid in users:
        recs = recommend_fn(int(uid), top_k=top_k)
        for r, wid in enumerate(recs, start=1):
            rows.append({"UserID": int(uid), "rank": int(r), "WineID": int(wid)})
    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_csv, index=False)
    print(f"Saved {len(out_df):,} rows -> {out_csv}")


In [29]:
export_arm_recs(
    recommend_fn=recommend_user_ner_ids,
    users=eval_users,
    out_csv=ARMS_DIR / "content_ner_tfidf_recs.csv",
    top_k=100
)


Saved 500,000 rows -> ../bandits/saved_arms/content_ner_tfidf_recs.csv
